# Unit 1: Heat — AI-Animated Video Generation (Open-Source Text-to-Video)

Class 6 Science, Term II (Tamil Nadu State Board). Same **AI Video Generation Pipeline** from
the Executive Architecture Design Document as the earlier Tamil rhyme notebook, re-run against
a new source chapter: Unit 1 "Heat" (sources of heat, what heat is, temperature, heat vs
temperature, thermal equilibrium, thermal expansion and its uses).

Differences from the earlier notebook:
- **English** narration and captions (matches the source text — no translation step), so the
  Tamil HarfBuzz/FreeType shaping isn't needed; plain Pillow text rendering is fine for Latin script.
- **Wan2.1-T2V-1.3B** open-source text-to-video model (Apache-2.0), with the 4-bit text-encoder
  fix for Colab's free-tier RAM ceiling.
- A final **compression stage** that targets a **5MB** output file via two-pass FFmpeg encoding.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.


## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Fail fast if there's no GPU, rather than discovering it after a 20GB download.
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected in this Colab runtime. Go to Runtime > Change runtime type > "
        "select a GPU (T4) > Save, then Runtime > Restart session, then run all cells again. "
        "If you already selected a GPU and still see this, you've likely hit Colab's "
        "free-tier GPU quota for now — wait a while and retry, or use Colab Pro."
    )
print("GPU OK:", torch.cuda.get_device_name(0))


In [ ]:
# Colab preinstalls MoviePy 1.x, where top-level `from moviepy import AudioFileClip` doesn't
# exist (that only works in MoviePy 2.x). `-U` forces the upgrade instead of a silent no-op.
!pip install -q -U diffusers transformers accelerate bitsandbytes imageio imageio-ffmpeg ftfy \
    edge-tts "moviepy>=2.0"


In [ ]:
import moviepy
from packaging.version import Version

if Version(moviepy.__version__) < Version("2.0"):
    raise RuntimeError(
        f"moviepy is still {moviepy.__version__} (need >=2.0). The pip install above "
        "upgraded the package on disk, but this kernel already had the old one loaded. "
        "Go to Runtime > Restart session (not Restart runtime and run all — just restart), "
        "then re-run every cell from the top. Do NOT re-run only the pip install cell alone."
    )
print("moviepy", moviepy.__version__, "OK")


In [ ]:
# A clean, readable font for English captions/titles (Latin script needs no complex shaping,
# unlike the Tamil version of this pipeline, so plain Pillow text rendering is fine here).
!wget -q -O Poppins-Bold.ttf "https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-Bold.ttf"
!wget -q -O Poppins-SemiBold.ttf "https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-SemiBold.ttf"
!ls -la Poppins-Bold.ttf Poppins-SemiBold.ttf


## 2. Caption / title rendering (plain Pillow — no HarfBuzz needed for English)

In [ ]:
from PIL import Image, ImageDraw, ImageFont

CAPTION_FONT = ImageFont.truetype("Poppins-SemiBold.ttf", 40)
TITLE_FONT = ImageFont.truetype("Poppins-Bold.ttf", 36)

TARGET_W, TARGET_H = 1280, 720


def wrap_text(text, font, max_width, draw):
    words = text.split(" ")
    lines, current = [], ""
    for w in words:
        trial = (current + " " + w).strip()
        box = draw.textbbox((0, 0), trial, font=font)
        if box[2] - box[0] > max_width and current:
            lines.append(current)
            current = w
        else:
            current = trial
    if current:
        lines.append(current)
    return lines


def caption_bar(text, font=CAPTION_FONT, max_width=1120, fg=(30, 20, 10, 255)):
    scratch = Image.new("RGBA", (10, 10), (0, 0, 0, 0))
    draw = ImageDraw.Draw(scratch)
    lines = wrap_text(text, font, max_width, draw)

    line_boxes = [draw.textbbox((0, 0), l, font=font) for l in lines]
    line_h = max(b[3] - b[1] for b in line_boxes) + 14
    pad_v, pad_h = 22, 34
    bar_h = pad_v * 2 + line_h * len(lines)
    bar_w = TARGET_W - 80

    bar = Image.new("RGBA", (bar_w, bar_h), (0, 0, 0, 0))
    bdraw = ImageDraw.Draw(bar)
    bdraw.rounded_rectangle([0, 0, bar_w, bar_h], radius=24, fill=(255, 250, 235, 235))

    y = pad_v
    for line, box in zip(lines, line_boxes):
        w = box[2] - box[0]
        x = (bar_w - w) // 2
        bdraw.text((x, y), line, font=font, fill=fg)
        y += line_h

    canvas = Image.new("RGBA", (TARGET_W, TARGET_H), (0, 0, 0, 0))
    canvas.paste(bar, (40, TARGET_H - bar_h - 36), bar)
    return canvas


def title_banner(text, font=TITLE_FONT, fg=(255, 255, 255, 255)):
    scratch = Image.new("RGBA", (10, 10), (0, 0, 0, 0))
    draw = ImageDraw.Draw(scratch)
    box = draw.textbbox((0, 0), text, font=font)
    w, h = box[2] - box[0], box[3] - box[1]
    pad_h, pad_v = 30, 16
    bar = Image.new("RGBA", (w + pad_h * 2, h + pad_v * 2), (0, 0, 0, 0))
    bdraw = ImageDraw.Draw(bar)
    bdraw.rounded_rectangle([0, 0, bar.width, bar.height], radius=18, fill=(30, 70, 130, 210))
    bdraw.text((pad_h - box[0], pad_v - box[1]), text, font=font, fill=fg)
    canvas = Image.new("RGBA", (TARGET_W, TARGET_H), (0, 0, 0, 0))
    canvas.paste(bar, ((TARGET_W - bar.width) // 2, 34), bar)
    return canvas


print("caption rendering OK")


## 3. Storyboard

Simplified from Unit 1 "Heat" (source pages 7-21): sources of heat, what heat is, hot/cold and
temperature, heat vs temperature, thermal equilibrium, thermal expansion, and its everyday uses —
matching the unit's own "Points to remember" summary. Each beat = one narration line + one AI
video sub-shot (kept generic/list-shaped in case a beat ever needs multiple sub-shots, same as
the rhyme notebook's monkey->fox split).


In [ ]:
STORY = {
    "chapter_title": "Unit 1: Heat",
    "voice": "en-IN-NeerjaNeural",
    "beats": [
        {"id": "intro",
         "narration": "Hi everyone! Today we're going to learn all about heat — what it is, how it moves, and why things change size when they get hot. Let's get started!",
         "clips": [{"prompt": "a bright sun radiating warm glowing heat waves over a green landscape, colorful educational cartoon animation style", "frames": 25, "hold": 1.0}]},
        {"id": "sources",
         "narration": "Heat comes from many places. The Sun is the biggest source of heat. We also get heat by rubbing two surfaces together, called friction, and from electricity flowing through wires.",
         "clips": [{"prompt": "hands rubbing together with warm glowing friction sparks next to a glowing red electric heater coil, colorful educational cartoon animation style", "frames": 21, "hold": 1.0}]},
        {"id": "what_is_heat",
         "narration": "Heat is a kind of energy. Every object is made of tiny particles called molecules, and they are always moving. When we heat something, its molecules vibrate and move faster.",
         "clips": [{"prompt": "glowing molecules inside a solid cube vibrating and moving faster as heat energy is added, simple educational science animation, colorful", "frames": 21, "hold": 1.0}]},
        {"id": "hot_cold_temperature",
         "narration": "We can feel if something is hot or cold, but our hands can trick us. To measure exactly how hot or cold something is, we use a thermometer, which measures temperature.",
         "clips": [{"prompt": "a cartoon thermometer dipped into a glass of hot water and a glass of cold water, red mercury rising and falling, colorful educational animation style", "frames": 21, "hold": 1.0}]},
        {"id": "heat_vs_temperature",
         "narration": "Heat and temperature are not the same thing. Temperature tells us how fast the molecules are moving. Heat depends on temperature and on how many molecules there are.",
         "clips": [{"prompt": "side by side comparison of a small steaming cup of tea and a large calm pond, educational infographic cartoon animation style, colorful", "frames": 21, "hold": 1.0}]},
        {"id": "thermal_equilibrium",
         "narration": "When a hot object touches a cold object, heat flows from the hot one to the cold one, until both reach the same temperature. This is called thermal equilibrium.",
         "clips": [{"prompt": "a glowing red hot object and a glowing blue cold object touching, warm energy flowing between them until both turn the same orange color, educational animation", "frames": 21, "hold": 1.0}]},
        {"id": "thermal_expansion",
         "narration": "Most things expand, or get bigger, when they are heated, and shrink back when they cool down. This is called thermal expansion.",
         "clips": [{"prompt": "a metal rod glowing orange and visibly stretching longer as it is heated over a flame, then shrinking back as it cools, simple educational cartoon animation", "frames": 21, "hold": 1.0}]},
        {"id": "uses_of_expansion",
         "narration": "Engineers use thermal expansion cleverly. They leave small gaps between railway tracks and bridge joints, so the metal doesn't crack when it expands in the heat.",
         "clips": [{"prompt": "a railway track with small visible gaps between the rails under bright sunshine, simple educational cartoon animation style, engineering diagram feel", "frames": 21, "hold": 1.0}]},
        {"id": "outro",
         "narration": "Now you know what heat is, how it flows, and why things expand when heated. Keep exploring the world of science around you. See you next time!",
         "clips": [{"prompt": "a happy cartoon classroom scene with a smiling student surrounded by icons of the sun, a thermometer, and a railway track, colorful warm animation style", "frames": 25, "hold": 1.0}]},
    ],
}
print(len(STORY["beats"]), "beats,", sum(len(b["clips"]) for b in STORY["beats"]), "AI clips total")


## 4. Load the open-source text-to-video model

`Wan-AI/Wan2.1-T2V-1.3B-Diffusers` (Apache-2.0). Same RAM fix as the rhyme notebook: the
UMT5-XXL text encoder (~5.7B params) is loaded in 4-bit — that's what actually caused the
"session crashed after using all available RAM" error on free-tier Colab, not the video model
itself, which is only ~2.6GB.


**Optional — cache on Google Drive.** Two separate things get cached here, because they're
very different sizes:

1. **The quantized text encoder (~2-3GB)** — after the first run, the 4-bit-quantized UMT5-XXL
   gets saved back to Drive, so every future run loads that small file instead of re-downloading
   the ~21GB full-precision original. This only needs ~5GB free and is worth doing almost always.
2. **The raw Hugging Face cache (~22-27GB, everything downloaded at full precision)** — optional,
   needs ~25GB free. Skip this if your Drive is tight; it only saves you from re-downloading the
   transformer+VAE (~5.7GB) on top of the text encoder, which is comparatively small anyway.

Skip this whole cell if you don't want to mount Drive at all.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

QUANT_TE_DIR = "/content/drive/MyDrive/wan2.1_text_encoder_4bit"  # small (~2-3GB) — check this first
DRIVE_CACHE_DIR = "/content/drive/MyDrive/wan2.1_model_cache"      # large (~22-27GB) — optional

QUANT_TE_REQUIRED_GB = 5
FULL_CACHE_REQUIRED_GB = 25

free_gb = shutil.disk_usage("/content/drive/MyDrive").free / (1024 ** 3)
print(f"Google Drive free space: {free_gb:.1f} GB")

if free_gb >= QUANT_TE_REQUIRED_GB:
    os.makedirs(QUANT_TE_DIR, exist_ok=True)
    print("Quantized text-encoder cache enabled at:", QUANT_TE_DIR)
else:
    QUANT_TE_DIR = None
    print(f"Only {free_gb:.1f} GB free — not even enough for the small quantized-encoder "
          f"cache ({QUANT_TE_REQUIRED_GB} GB). It will be re-downloaded and re-quantized every session.")

if free_gb >= FULL_CACHE_REQUIRED_GB:
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ["HF_HOME"] = DRIVE_CACHE_DIR
    print("Full raw Hugging Face cache also redirected to Drive:", DRIVE_CACHE_DIR)
else:
    os.environ.pop("HF_HOME", None)  # use huggingface_hub's default local (non-persistent) cache
    print(f"Not enough free space for the full {FULL_CACHE_REQUIRED_GB}GB raw cache — "
          "transformer/VAE will use the Colab VM's local disk instead (fine, they're small).")


In [ ]:
import os
import gc

# hf_transfer is deprecated (huggingface_hub now uses "Xet" for fast transfer by default,
# no env var needed) — nothing to set here anymore, downloads are already fast out of the box.

import torch
from transformers import UMT5EncoderModel, BitsAndBytesConfig
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
WAN_W, WAN_H = 640, 368
WAN_FPS = 16

# QUANT_TE_DIR is set by the optional Drive-caching cell above; define it here too in case
# that cell was skipped, so this cell still runs standalone (just without persistence).
if "QUANT_TE_DIR" not in dir():
    QUANT_TE_DIR = None

quant_te_cached = QUANT_TE_DIR and os.path.exists(os.path.join(QUANT_TE_DIR, "config.json"))

if quant_te_cached:
    print("Loading previously-quantized text encoder from Drive cache (fast, ~2-3GB, no big re-download)...")
    text_encoder = UMT5EncoderModel.from_pretrained(
        QUANT_TE_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    )
else:
    print("No cached quantized text encoder found — downloading full-precision UMT5-XXL once (~21GB)...")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    )
    text_encoder = UMT5EncoderModel.from_pretrained(
        MODEL_ID, subfolder="text_encoder", quantization_config=quant_config, torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    if QUANT_TE_DIR:
        os.makedirs(QUANT_TE_DIR, exist_ok=True)
        text_encoder.save_pretrained(QUANT_TE_DIR)
        print("Saved quantized text encoder to Drive for next time:", QUANT_TE_DIR)

print("Loading VAE + transformer...")
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(
    MODEL_ID, vae=vae, text_encoder=text_encoder, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
gc.collect()
torch.cuda.empty_cache()

NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, extra limbs, bad anatomy, "
    "static image, watermark, text, subtitles, worst quality, jpeg artifacts, ugly"
)

print("model loaded:", MODEL_ID)


## 5. Generate an AI video clip per beat (Video Rendering agent)

Each clip is short (33-41 frames, ~2-2.5s) and gets looped/trimmed in Assembly to fill its
beat's actual narration length. Expect a few minutes per clip on a T4; 9 clips total.


In [ ]:
os.makedirs("clips", exist_ok=True)
CLIP_PATHS = {}


def generate_with_retry(prompt, num_frames, width=WAN_W, height=WAN_H, steps=30, attempt=0):
    """640x368 already has a large safety margin under a T4's ~15GB, but GPUs get shared /
    fragmented unpredictably on free tiers — if it OOMs anyway, halve resolution once and retry
    rather than losing the whole session."""
    try:
        output = pipe(
            prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
            height=height, width=width, num_frames=num_frames,
            guidance_scale=5.0, num_inference_steps=steps,
        )
        return output.frames[0]
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        if attempt >= 1:
            raise
        half_w = max(16, (width // 2) // 16 * 16)
        half_h = max(16, (height // 2) // 16 * 16)
        print(f"  OOM at {width}x{height} — retrying once at {half_w}x{half_h}...")
        return generate_with_retry(prompt, num_frames, half_w, half_h, steps, attempt + 1)


for beat in STORY["beats"]:
    for i, shot in enumerate(beat["clips"]):
        out_path = f"clips/{beat['id']}_{i}.mp4"
        print("generating:", beat["id"], i, "->", shot["prompt"])
        video_frames = generate_with_retry(shot["prompt"], shot["frames"])
        export_to_video(video_frames, out_path, fps=WAN_FPS)
        CLIP_PATHS[(beat["id"], i)] = out_path

        del video_frames
        gc.collect()
        torch.cuda.empty_cache()

print("done:", len(CLIP_PATHS), "clips")


## 6. Narration audio (Audio & Sync stage — Edge TTS, Indian-English voice)

In [ ]:
import asyncio
import edge_tts

os.makedirs("audio", exist_ok=True)
AUDIO_PATHS = {}

async def synth_all():
    for beat in STORY["beats"]:
        mp3_path = f"audio/{beat['id']}.mp3"
        communicate = edge_tts.Communicate(beat["narration"], STORY["voice"], rate="-5%")
        await communicate.save(mp3_path)
        AUDIO_PATHS[beat["id"]] = mp3_path
        print("synthesized:", beat["id"])

await synth_all()


## 7. Assembly (Video Rendering + Publishing agents)

Same shape as the rhyme notebook: split each beat's narration duration across its clips by
`hold`, loop/trim each AI clip to fill its share, letterbox to 1280x720, overlay the caption bar
(and a title banner on intro/outro), concatenate every beat, mux narration audio, write MP4 + SRT.


In [ ]:
from moviepy import (
    AudioFileClip, VideoFileClip, VideoClip, concatenate_videoclips, concatenate_audioclips,
)
import numpy as np

FPS = 24


def fit_letterbox(get_frame_fn, src_w, src_h):
    scale = min(TARGET_W / src_w, TARGET_H / src_h)
    new_w, new_h = int(src_w * scale), int(src_h * scale)
    x_off, y_off = (TARGET_W - new_w) // 2, (TARGET_H - new_h) // 2

    def make_frame(t):
        frame = Image.fromarray(get_frame_fn(t)).resize((new_w, new_h), Image.LANCZOS)
        canvas = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)
        canvas[y_off:y_off + new_h, x_off:x_off + new_w] = np.array(frame)
        return canvas

    return make_frame


def looped_clip(path, duration):
    clip = VideoFileClip(path)
    src_w, src_h = clip.size
    frame_fn = fit_letterbox(lambda t: clip.get_frame(t % clip.duration), src_w, src_h)
    return VideoClip(frame_fn, duration=duration)


def overlay_rgba(base_clip, rgba_img, duration):
    arr = np.array(rgba_img)
    rgb = arr[:, :, :3].astype(np.float32)
    alpha = (arr[:, :, 3:4].astype(np.float32)) / 255.0

    def make_frame(t):
        under = base_clip.get_frame(t).astype(np.float32)
        comp = rgb * alpha + under * (1 - alpha)
        return comp.astype(np.uint8)

    return VideoClip(make_frame, duration=duration)


def srt_timestamp(seconds):
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


audio_clips_ordered = []
video_clips_ordered = []
srt_entries = []
t_cursor = 0.0

for beat in STORY["beats"]:
    bid = beat["id"]
    mp3_path = AUDIO_PATHS.get(bid, f"audio/{bid}.mp3")
    audio_clip = AudioFileClip(mp3_path)
    dur = audio_clip.duration
    audio_clips_ordered.append(audio_clip)

    hold_sum = sum(shot["hold"] for shot in beat["clips"])
    sub_clips = []
    for i, shot in enumerate(beat["clips"]):
        shot_dur = dur * (shot["hold"] / hold_sum)
        clip_path = CLIP_PATHS.get((bid, i), f"clips/{bid}_{i}.mp4")
        sub_clips.append(looped_clip(clip_path, shot_dur))
    visual = concatenate_videoclips(sub_clips) if len(sub_clips) > 1 else sub_clips[0]

    cap = caption_bar(beat["narration"])
    visual = overlay_rgba(visual, cap, dur)
    if bid in ("intro", "outro"):
        banner = title_banner(STORY["chapter_title"])
        visual = overlay_rgba(visual, banner, dur)

    video_clips_ordered.append(visual)
    srt_entries.append((t_cursor, t_cursor + dur, beat["narration"]))
    t_cursor += dur

final_video = concatenate_videoclips(video_clips_ordered, method="compose")
final_audio = concatenate_audioclips(audio_clips_ordered)
final_video = final_video.with_audio(final_audio)

os.makedirs("output", exist_ok=True)
raw_path = "output/heat_unit_raw.mp4"
final_video.write_videofile(
    raw_path, fps=FPS, codec="libx264", audio_codec="aac", preset="medium",
    ffmpeg_params=["-movflags", "+faststart", "-crf", "20"],
)

srt_path = "output/heat_unit.srt"
with open(srt_path, "w", encoding="utf-8") as f:
    for i, (start, end, text) in enumerate(srt_entries, 1):
        f.write(f"{i}\n{srt_timestamp(start)} --> {srt_timestamp(end)}\n{text}\n\n")

print("Raw video:", raw_path)
print("Subtitles:", srt_path)
print("Total duration:", round(t_cursor, 2), "s")


## 8. Compress to <=5MB

Two-pass FFmpeg encode targeting a bitrate computed from the actual duration, so the output
lands right at the 5MB budget instead of guessing a single CRF value. Targets 4.8MB internally
to leave headroom for container/muxing overhead so the final file stays under the 5MB requirement.


In [ ]:
import subprocess

TARGET_MB = 4.8
AUDIO_KBPS = 64
duration = final_video.duration

target_total_kbps = (TARGET_MB * 8192) / duration  # 1 MB = 8192 kilobits
video_kbps = max(int(target_total_kbps - AUDIO_KBPS), 150)
print(f"duration={duration:.1f}s target_total={target_total_kbps:.0f}kbps video_kbps={video_kbps}")

compressed_path = "output/heat_unit_compressed.mp4"

subprocess.run([
    "ffmpeg", "-y", "-i", raw_path, "-c:v", "libx264", "-b:v", f"{video_kbps}k",
    "-pass", "1", "-an", "-f", "mp4", "/dev/null",
], check=True)
subprocess.run([
    "ffmpeg", "-y", "-i", raw_path, "-c:v", "libx264", "-b:v", f"{video_kbps}k",
    "-pass", "2", "-c:a", "aac", "-b:a", f"{AUDIO_KBPS}k", "-movflags", "+faststart",
    compressed_path,
], check=True)

size_mb = os.path.getsize(compressed_path) / (1024 * 1024)
print(f"compressed size: {size_mb:.2f} MB -> {compressed_path}")
if size_mb > 5.0:
    print("WARNING: still above 5MB — lower TARGET_MB above and re-run this cell, or drop WAN_W/WAN_H.")
else:
    print("Under 5MB target.")


## 9. Download the result

In [ ]:
from google.colab import files
files.download(compressed_path)
files.download(srt_path)
